In [178]:
import pandas as pd

In [179]:
!pip install xlrd

In [180]:
url = "C:/Users/yurii/Downloads/IH_Projects/shark_attack/GSAF5.xls"

shark_attack = pd.read_excel("C:/Users/yurii/Downloads/IH_Projects/shark_attack/GSAF5.xls")

In [181]:
#show columns

shark_attack.columns

Index(['Date', 'Year', 'Type', 'Country', 'State', 'Location', 'Activity',
       'Name', 'Sex', 'Age', 'Injury', 'Fatal Y/N', 'Time', 'Species ',
       'Source', 'pdf', 'href formula', 'href', 'Case Number', 'Case Number.1',
       'original order', 'Unnamed: 21', 'Unnamed: 22'],
      dtype='str')

In [182]:
#drop the columns we don't need

shark_attack = shark_attack.drop(
    columns=[
        'Name', 'Source', 'pdf', 'href formula', 'href',
        'Case Number', 'Case Number.1', 'original order',
        'Unnamed: 21', 'Unnamed: 22'
    ]
)

In [183]:
#drop the rows older than 1975 (left 50 years)
shark_attack = shark_attack[shark_attack['Year'] > 1975]

In [184]:
shark_attack

,Date,Year,Type,Country,State,Location,Activity,Sex,Age,Injury,Fatal Y/N,Time,Species
0,18th September,2026.0,Unprovoked,Australia,Western Australia,Sorrento Beach Perth,Swimming,M,63,Body not recovered,Y,1015hrs,Great White Shark 6m (20ft)
1,16th September,2026.0,Unprovoked,Canada,Quebec,Off the coast of Perce Le Bilbo dive site,Diving,M,?,Injuries to chest shoulder and back,N,1030hrs,Great White Shark
2,14th September,2026.0,Unprovoked,Bahamas,Bimini,Bimini Island,Swimming,F,37,Serious injuries to right ankle,N,1730hrs,Unknown
3,13th September,2026.0,Unprovoked,Australia,Western Australia,Geraldton,Surfing,M,50's,Foot lost in attack,N,0945hrs,Unknown
4,7th September,2026.0,Unprovoked,USA,Hawaii,Honolulu,Surfing,M,24,Scrapes on inside of legs,N,1640hrs,Tiger Shark suspected
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,12-Jan-1976,1976.0,Unprovoked,AUSTRALIA,Queensland,Harvey Bay,NaN,F,NaN,Survived,N,NaN,NaN
3996,11-Jan-1976,1976.0,Provoked,SOUTH AFRICA,Western Cape Province,Kalk Bay,Fishing for snoek & yellowtail,NaN,NaN,"Hooked shark leapt onboard & into fish well, w...",N,07h30,"White shark, 3 m [10']"
3997,08-Jan-1976,1976.0,Unprovoked,USA,Florida,"Off Fort Pierce, St Lucie County",Spearfishing / scuba diving,M,25,Puncture wounds to head & neck,N,NaN,6' shark
3998,02-Jan-1976,1976.0,Unprovoked,NEW ZEALAND,North Island,"Te Kaha, East coast",Spearfishing,M,NaN,FATAL,Y,13h00,Bronze whaler shark


In [185]:
# Select only the columns I want to clean
columns_to_clean = ["Country", "State", "Location", "Activity"]

shark_clean = shark_attack[columns_to_clean].copy()

In [186]:
print(shark_clean.isna().sum())

Country       3
State       166
Location    172
Activity    211
dtype: int64


In [187]:
# Remove extra spaces
for column in columns_to_clean:
    shark_clean[column] = shark_clean[column].str.strip()
display(shark_clean.head())

,Country,State,Location,Activity
0,Australia,Western Australia,Sorrento Beach Perth,Swimming
1,Canada,Quebec,Off the coast of Perce Le Bilbo dive site,Diving
2,Bahamas,Bimini,Bimini Island,Swimming
3,Australia,Western Australia,Geraldton,Surfing
4,USA,Hawaii,Honolulu,Surfing


In [188]:
# Replace empty strings with NaN
shark_clean = shark_clean.replace(r"^\s*$", pd.NA, regex=True)

In [189]:
# Standardize capitalization
shark_clean["Country"] = shark_clean["Country"].str.title()
shark_clean["State"] = shark_clean["State"].str.title()
shark_clean["Location"] = shark_clean["Location"].str.title()

In [190]:
# For Activity, remove extra spaces
shark_clean["Activity"] = shark_clean["Activity"].str.strip()

In [191]:
# Check missing values again
print(shark_clean.isna().sum())

Country       3
State       166
Location    172
Activity    212
dtype: int64


In [192]:
# Check cleaned data
display(shark_clean.head(20))

,Country,State,Location,Activity
0,Australia,Western Australia,Sorrento Beach Perth,Swimming
1,Canada,Quebec,Off The Coast Of Perce Le Bilbo Dive Site,Diving
2,Bahamas,Bimini,Bimini Island,Swimming
3,Australia,Western Australia,Geraldton,Surfing
4,Usa,Hawaii,Honolulu,Surfing
5,Usa,Florida,New Smyrna Beach,Surfing
6,Usa,Florida,30 Miles Off Pensacola,Spearfishing
7,Usa,Massachusetts,Norton Point Edgartown Marthas Vinyard,Swimming
8,Usa,Hawaii,"Ala Moana Beach Park, Oahu",Surfing
9,Usa,Florida,Boca Grande (Island In The Keys),Wading on a sandbar


In [193]:
# Select only the columns I want to clean (y = Yurii)
columns_to_clean_y = ['Sex', 'Age', 'Injury']

shark_clean_y = shark_attack[columns_to_clean_y].copy()

shark_clean_y


,Sex,Age,Injury
0,M,63,Body not recovered
1,M,?,Injuries to chest shoulder and back
2,F,37,Serious injuries to right ankle
3,M,50's,Foot lost in attack
4,M,24,Scrapes on inside of legs
...,...,...,...
3995,F,NaN,Survived
3996,NaN,NaN,"Hooked shark leapt onboard & into fish well, w..."
3997,M,25,Puncture wounds to head & neck
3998,M,NaN,FATAL


In [194]:
# cleaning "Sex"

shark_clean_y["Sex"] = shark_clean_y["Sex"].str.strip().str.upper()

shark_clean_y["Sex"] = shark_clean_y["Sex"].fillna("Unknown")

shark_clean_y.loc[
    ~shark_clean_y["Sex"].isin(["M", "F"]),
    "Sex"
] = "Unknown"

In [195]:
# checking "Sex"

shark_clean_y["Sex"].value_counts(dropna=False)

Sex
M          3121
F           646
Unknown     231
Name: count, dtype: int64

In [196]:
# Cleaning "Age". Convert valid ages to numbers and replace unclear/non-numeric values with NaN

shark_clean_y["Age"] = pd.to_numeric(
    shark_clean_y["Age"],
    errors="coerce"
)

In [197]:
# checking "Age"

shark_clean_y["Age"].unique()

array([63., nan, 37., 24., 40., 43., 27., 71., 21., 69., 31., 13., 12.,
       17., 35., 19., 11., 20., 38., 39., 16., 22., 55., 26., 56., 25.,
       61., 14., 54., 48., 57.,  8.,  9.,  7., 85., 18., 66., 42., 45.,
       30., 60., 29., 58., 36., 23., 28., 68., 33., 15., 41., 49., 46.,
       65., 64., 32., 10., 62., 52., 44., 47., 59., 50., 34., 77., 73.,
       67.,  6., 53., 51., 75., 70.,  4., 74.,  3., 82., 72.,  5., 86.,
       84., 87.])

In [198]:
# Cleanign "Injury". Standardize Injury text for keyword-based categorization

shark_clean_y["Injury_clean"] = (
    shark_clean_y["Injury"]
    .str.strip()
    .str.lower()
)

In [199]:
# checking key-words to create new injury categories

keywords = [
    "fatal",
    "bite",
    "bitten",
    "wound",
    "puncture",
    "laceration",
    "lost",
    "amput",
    "minor",
    "scratch",
    "abrasion",
    "no injury",
    "uninjured",
    "survived"
]

for word in keywords:
    count = shark_clean_y["Injury_clean"].str.contains(
        word,
        case=False,
        na=False
    ).sum()
    
    print(word, count)

fatal 403
bite 128
bitten 1020
wound 209
puncture 240
laceration 777
lost 8
amput 36
minor 284
scratch 18
abrasion 30
no injury 541
uninjured 2
survived 52


In [200]:
# Categorize injury descriptions based on keywords

def categorize_injury(text):
    if pd.isna(text):
        return "Unknown"

    if any(word in text for word in [
        "no details", "not stated", "unknown", "none to report"
    ]):
        return "Unknown"

    if "fatal" in text:
        return "Fatal"

    elif "no injury" in text or "uninjured" in text:
        return "No injury"

    elif any(word in text for word in [
        "amput", "sever", "lost", "torn off",
        "stripped of flesh", "tissue loss"
    ]):
        return "Severe injury"

    elif any(word in text for word in [
        "minor", "scratch", "abrasion"
    ]):
        return "Minor injury"

    elif any(word in text for word in [
        "bite", "bitten", "wound", "puncture",
        "lacerat", "injur", "gash", "cut"
    ]):
        return "Injury"

    else:
        return "Other"

In [201]:
# making a new column "Injury_Category" (instead of old "Injury")

shark_clean_y["Injury_Category"] = shark_clean_y["Injury_clean"].apply(categorize_injury)

In [202]:
# Final injury categories

shark_clean_y["Injury_Category"].value_counts(dropna=False)

Injury_Category
Injury           2222
No injury         543
Fatal             402
Minor injury      316
Other             265
Severe injury     205
Unknown            45
Name: count, dtype: int64

In [203]:
# Keep only the final cleaned columns

shark_clean_y = shark_clean_y[
    ["Sex", "Age", "Injury_Category"]
].copy()

In [204]:
# checking the result of my cleaning

shark_clean_y.head(20)

,Sex,Age,Injury_Category
0,M,63.0,Other
1,M,NaN,Injury
2,F,37.0,Injury
3,M,NaN,Severe injury
4,M,24.0,Other
5,M,NaN,Injury
6,M,40.0,Injury
7,F,NaN,Injury
8,F,NaN,No injury
9,M,43.0,Injury


# Cleaning data values for Fatal, Species and Time columns

## Fatal Y/N

In [205]:
# checking all entries for a column Fatal
shark_attack['Fatal Y/N'].value_counts()

Fatal Y/N
N          3258
Y           438
UNKNOWN      24
F             5
M             3
n             1
Nq            1
2017          1
Y x 2         1
Name: count, dtype: int64

In [206]:
# cleaning values for Fatal Y/N to have only: Y, N, UNKNOWN
shark_attack['Fatal Y/N'] = shark_attack['Fatal Y/N'].replace(["F", "M", "Y x 2"], "Y")

In [207]:
shark_attack['Fatal Y/N'] = shark_attack['Fatal Y/N'].replace(["n", "Nq"], "N")

In [208]:
# cleaning values for Time, first checking how many unique values we have
shark_attack['Time'].nunique()

434

In [209]:
# Extract and replace digits using RegEx and converts to a string. \D+ for all strings that do not contain digits or characters

cleaned = shark_attack['Time'].astype(str).str.replace(r'\D+', '', regex=True)

In [210]:
# See the distribution of digit counts
print(cleaned.str.len().value_counts())

Time
4.0     2228
0.0      437
8.0       16
3.0        5
6.0        2
1.0        2
2.0        2
5.0        2
10.0       1
Name: count, dtype: int64


In [211]:
#removing values with 8 digits, after a first check, they could be replaced to first 4 digits
cleaned = cleaned.mask(cleaned.str.len() == 8, cleaned.str[:4])

In [212]:
#check again the distribution, the 3-10 digits due to low number can be ignored
print(cleaned.str.len().value_counts())

Time
4.0     2244
0.0      437
3.0        5
6.0        2
1.0        2
2.0        2
5.0        2
10.0       1
Name: count, dtype: int64


In [213]:
# Keeping only valid 4-digit strings, convert everything else to NaN, and format to HH:MM
shark_attack['Time'] = pd.to_datetime(
    cleaned.where(cleaned.str.len() == 4),
    format='%H%M',
    errors='coerce'
).dt.strftime('%H:%M')

In [214]:
# Checking the column entries
shark_attack['Time'].head(20)

0     10:15
1     10:30
2     17:30
3     09:45
4     16:40
5     11:00
6     16:35
7     13:40
8     13:30
9     17:00
10      NaN
11    16:15
12    13:00
13    17:15
14    17:30
15      NaN
16    17:00
17      NaN
18    09:00
19      NaN
Name: Time, dtype: str

## Species

In [215]:
shark_attack.columns

Index(['Date', 'Year', 'Type', 'Country', 'State', 'Location', 'Activity',
       'Sex', 'Age', 'Injury', 'Fatal Y/N', 'Time', 'Species '],
      dtype='str')

In [216]:
#fixing Species column name by removing the space
shark_attack.rename(columns = {'Species ': 'Species'}, inplace = True)

In [217]:
# checking most mentioned shark types
shark_attack['Species'].value_counts().head(30)

Species
White shark                                           137
Shark involvement not confirmed                        73
Bull shark                                             62
Tiger shark                                            62
Shark involvement prior to death was not confirmed     52
Invalid                                                42
4' shark                                               37
6' shark                                               28
4' to 5' shark                                         25
Blacktip shark                                         22
5' shark                                               22
Unknown                                                21
3' shark                                               21
2 m shark                                              21
No shark involvement                                   21
3' to 4' shark                                         20
Wobbegong shark                                        20
1.2 m 

In [218]:
import numpy as np

# Define conditions for the most mentioned shark types
conditions = [
    shark_attack['Species'].str.contains('white', case=False, na=False),
    shark_attack['Species'].str.contains('tiger', case=False, na=False),
    shark_attack['Species'].str.contains('bull', case=False, na=False),
    shark_attack['Species'].str.contains('nurse', case=False, na=False),
    shark_attack['Species'].str.contains('blacktip', case=False, na=False),
    shark_attack['Species'].str.contains('wobbegong', case=False, na=False),
    shark_attack['Species'].str.contains('raggedtooth', case=False, na=False),
    shark_attack['Species'].str.contains('lemon', case=False, na=False),
    shark_attack['Species'].str.contains('bronze whaler', case=False, na=False),
]

# Define corresponding standard names
choices = [
    'White Shark',
    'Tiger Shark',
    'Bull Shark',
    'Nurse Shark',
    'Blacktip Shark',
    'Wobbegong Shark',
    'Raggedtooth Shark',
    'Lemon Shark',
    'Bronze Whaler Shark',
]

# Assign to an existing column (anything else becomes 'Other/Unknown'), with that we remove the null values as well
shark_attack['Species'] = np.select(conditions, choices, default='Other/Unknown')

# Cleaning Null Values for Fatal and Time columns

In [219]:
# Count the number of null values in both columns
shark_attack[["Fatal Y/N", "Time"]].isna().sum()

Fatal Y/N     266
Time         1754
dtype: int64

In [220]:
# renaming all Null values
shark_attack["Time"] = shark_attack["Time"].fillna("No time specified")

In [221]:
shark_attack["Fatal Y/N"] = shark_attack["Fatal Y/N"].fillna("UNKNOWN")

In [222]:
shark_attack["Fatal Y/N"].value_counts()

Fatal Y/N
N          3260
Y           447
UNKNOWN     290
2017          1
Name: count, dtype: int64

In [223]:
# for some reason there was "2017" that could not be located by using ==, therefore, it was be removed by using .contains()
shark_attack['Fatal Y/N'] = shark_attack['Fatal Y/N'].astype(str).str.strip().replace({'2017': 'UNKNOWN'})

In [224]:
#final check if we missed any null values
shark_attack[["Time","Species","Fatal Y/N"]].isnull().sum()

Time         0
Species      0
Fatal Y/N    0
dtype: int64

In [225]:
shark_clean_tfs = shark_attack[["Time", "Species", "Fatal Y/N"]]

In [226]:
shark_clean_tfs.head(10)

,Time,Species,Fatal Y/N
0,10:15,White Shark,Y
1,10:30,White Shark,N
2,17:30,Other/Unknown,N
3,09:45,Other/Unknown,N
4,16:40,Tiger Shark,N
5,11:00,Other/Unknown,N
6,16:35,Bull Shark,N
7,13:40,Other/Unknown,N
8,13:30,Tiger Shark,N
9,17:00,Other/Unknown,N


In [227]:
# Add Date, Year and Type to the cleaned DataFrame
shark_clean[["Date", "Year", "Type"]] = shark_attack[["Date", "Year", "Type"]]

In [228]:
display(shark_clean[["Date", "Year", "Type"]].head(50))

,Date,Year,Type
0,18th September,2026.0,Unprovoked
1,16th September,2026.0,Unprovoked
2,14th September,2026.0,Unprovoked
3,13th September,2026.0,Unprovoked
4,7th September,2026.0,Unprovoked
5,2nd September,2026.0,Unprovoked
6,1st September,2026.0,Unprovoked
7,31st August,2026.0,Unprovoked
8,26th August,2026.0,Unprovoked
9,23rd August,2026.0,Unprovoked


In [229]:
#Convert "Year"" to integers
shark_clean["Year"] = pd.to_numeric(
    shark_clean["Year"],
    errors="coerce"
).astype("Int64")

In [230]:
#Check "Year"
shark_clean["Year"] = pd.to_numeric(shark_clean["Year"],errors="coerce").astype("Int64")

In [231]:
print(shark_clean["Year"].head())
print(shark_clean["Year"].dtype)
print("Missing years:", shark_clean["Year"].isna().sum())

0    2026
1    2026
2    2026
3    2026
4    2026
Name: Year, dtype: Int64
Int64
Missing years: 0


In [232]:
#Clean "Type"
shark_clean["Type"] = shark_clean["Type"].str.strip()
shark_clean["Type"] = shark_clean["Type"].replace("", pd.NA)
shark_clean["Type"] = shark_clean["Type"].str.title()

In [233]:
#Check the data types cleanned
print(shark_clean["Type"].value_counts(dropna=False))

Type
Unprovoked             3162
Provoked                304
Invalid                 266
Watercraft              176
Sea Disaster             45
Questionable             27
NaN                      14
?                         1
Unconfirmed               1
Unverified                1
Under Investigation       1
Name: count, dtype: int64


In [234]:
#Clean Date as text
shark_clean["Date"] = shark_clean["Date"].str.strip()
shark_clean["Date"] = shark_clean["Date"].replace("", pd.NA)

In [235]:
#Remove ordinal suffixes
shark_clean["Date"] = shark_clean["Date"].str.replace(r"(\d+)(st|nd|rd|th)",r"\1",regex=True)

In [236]:
print(shark_clean["Date"].head(20))

0     18 September
1     16 September
2     14 September
3     13 September
4      7 September
5      2 September
6      1 September
7        31 August
8        26 August
9        23 August
10       19 August
11       11 August
12        8 August
13         29 July
14         25 July
15         24 July
16         24 July
17         21 July
18         18 July
19          3 July
Name: Date, dtype: object


In [237]:
#Check date

print(shark_clean["Date"].sample(50).to_string())

874         Sep-2017
2805     11-Sep-2000
2164     17-Jun-2007
3585     28-Feb-1987
1334     18-Jun-2014
1681    11-Aug--2011
3441     05-Mar-1990
237      04 Jul-2023
3205     09-Dec-1994
2026     19-Jul-2008
2118     27-Sep-2007
2896     05-Sep-1999
2006     30-Aug-2008
3389       July 1991
1874     19-Oct-2009
1270     16-Nov-2014
2614     03-Oct-2002
493      13-Jan-2021
260      05-May-2023
1026     04-Aug-2016
284      04-Feb-2023
2505     26-Dec-2003
149              NaN
1077     22-Apr-2016
2290     18-Mar-2006
2980     29-May-1998
1292     13-Sep-2014
1080     18-Apr-2016
1601     14-Mar-2012
3352     28-Mar-1992
3552     14-Jan-1988
2607     02-Nov-2002
2579     20-Apr-2003
3469     09-Aug-1989
1769     23-Oct-2010
3964     26-Nov-1976
595      13-Jan-2020
3569     21-Jul-1987
1472     28-Apr-2013
981      27-Dec-2016
3596     04-Dec-1986
1608     01-Mar-2012
1844     12-Jan-2010
414      13-Aug-2021
3028     11-Aug-1997
935      29-Apr-2017
2230     02-Sep-2006
2550     10-J

In [238]:
print("Missing dates:", shark_clean["Date"].isna().sum())

Missing dates: 80


In [239]:
#after check, we can see that some of the dates are not in a standard format. We will need to clean this column further to ensure consistency and accuracy in our analysis.
#we mainly need the date for month/season analysis, while Year can handle trends over time.
#Rather than heavily modifying Date, I'd preserve it and create a new Month column.



In [240]:
#Formating Date column
shark_clean["Date"] = shark_clean["Date"].str.lower()
shark_clean["Date"] = shark_clean["Date"].str.replace("out", "oct", regex=False)
print(shark_clean["Date"].sample(100).to_string())

2043             07-jun-2008
931     reported 06-may-2017
3949             26-may-1977
3077             02-sep-1996
3886             01-dec-1979
67               10 november
3880             summer 1980
456              15-may-2021
2738             23-may-2001
895              29-jul-2017
1141             09-oct-2015
3183             12-may-1995
55                13 january
2436             29-aug-2004
2673             03-mar-2002
1776             01-oct-2010
1395             10-nov-2013
2801             16-sep-2000
2954             01-oct-1998
2360             22-jul-2005
288              05-jan-2023
3234    reported 16-apr-1994
3184             16-apr-1995
1433             13-aug-2013
2834             04-jul-2000
1127             10-nov-2015
2695             15-sep-2001
241              03 jul-2023
952              05-apr-2017
1954    reported 27-jan-2009
3576             25-may-1987
1414             21-sep-2013
1744             28-jan-2011
3801             13-dec-1981
1999          

In [241]:
#Create new column "Month" based on the "Date" column
shark_clean["Month"] = pd.NA

In [242]:
shark_clean[["Date", "Year", "Month"]].head(100)

,Date,Year,Month
0,18 september,2026,<NA>
1,16 september,2026,<NA>
2,14 september,2026,<NA>
3,13 september,2026,<NA>
4,7 september,2026,<NA>
...,...,...,...
95,29 june,2025,<NA>
96,25 june,2025,<NA>
97,22 june,2025,<NA>
98,17 june,2025,<NA>


In [243]:
shark_clean["Date"].str.contains("jan", case=False, na=False)

0       False
1       False
2       False
3       False
4       False
        ...  
3995     True
3996     True
3997     True
3998     True
3999    False
Name: Date, Length: 3998, dtype: bool

In [244]:
shark_clean.loc[shark_clean["Date"].str.contains("jan", case=False, na=False),"Month"] = "January"
shark_clean.loc[shark_clean["Date"].str.contains("feb", case=False, na=False),"Month"] = "February"
shark_clean.loc[shark_clean["Date"].str.contains("mar", case=False, na=False),"Month"] = "March"
shark_clean.loc[shark_clean["Date"].str.contains("apr", case=False, na=False),"Month"] = "April"
shark_clean.loc[shark_clean["Date"].str.contains("may", case=False, na=False),"Month"] = "May"
shark_clean.loc[shark_clean["Date"].str.contains("jun", case=False, na=False),"Month"] = "June"
shark_clean.loc[shark_clean["Date"].str.contains("jul", case=False, na=False),"Month"] = "July"
shark_clean.loc[shark_clean["Date"].str.contains("aug", case=False, na=False),"Month"] = "August"
shark_clean.loc[shark_clean["Date"].str.contains("sep", case=False, na=False),"Month"] = "September"
shark_clean.loc[shark_clean["Date"].str.contains("oct", case=False, na=False),"Month"] = "October"
shark_clean.loc[shark_clean["Date"].str.contains("nov", case=False, na=False),"Month"] = "November"
shark_clean.loc[shark_clean["Date"].str.contains("dec", case=False, na=False),"Month"] = "December"

In [245]:
#check Month column cleanned
shark_clean["Month"].value_counts(dropna=False)

Month
July         488
September    396
August       396
June         357
October      334
April        298
May          296
January      287
March        280
November     259
December     253
February     228
<NA>         126
Name: count, dtype: int64

In [246]:
#Check the "NA" values in the "Month" column 
shark_clean.loc[shark_clean["Month"].isna(),["Date", "Year", "Month"]].head(50)

,Date,Year,Month
99,NaN,2025,<NA>
101,NaN,2025,<NA>
102,NaN,2025,<NA>
103,NaN,2025,<NA>
104,NaN,2025,<NA>
105,NaN,2025,<NA>
106,NaN,2025,<NA>
107,NaN,2025,<NA>
108,NaN,2025,<NA>
110,NaN,2025,<NA>


In [247]:
#Fixing the portuguese month names
shark_clean.loc[shark_clean["Date"].str.contains("fev", case=False, na=False),"Month"] = "February"
shark_clean.loc[shark_clean["Date"].str.contains("mar", case=False, na=False),"Month"] = "March"
shark_clean.loc[shark_clean["Date"].str.contains("abr", case=False, na=False),"Month"] = "April"
shark_clean.loc[shark_clean["Date"].str.contains("mai", case=False, na=False),"Month"] = "May"
shark_clean.loc[shark_clean["Date"].str.contains("jun", case=False, na=False),"Month"] = "June"
shark_clean.loc[shark_clean["Date"].str.contains("jul", case=False, na=False),"Month"] = "July"
shark_clean.loc[shark_clean["Date"].str.contains("ago", case=False, na=False),"Month"] = "August"
shark_clean.loc[shark_clean["Date"].str.contains("set", case=False, na=False),"Month"] = "September"
shark_clean.loc[shark_clean["Date"].str.contains("out", case=False, na=False),"Month"] = "October"
shark_clean.loc[shark_clean["Date"].str.contains("nov", case=False, na=False),"Month"] = "November"
shark_clean.loc[shark_clean["Date"].str.contains("dez", case=False, na=False),"Month"] = "December"

In [248]:
#Check the "Month" column after formating
shark_clean["Month"].value_counts(dropna=False)

Month
July         488
September    396
August       396
June         357
October      334
April        298
May          296
January      287
March        280
November     259
December     253
February     228
<NA>         126
Name: count, dtype: int64

In [249]:
#Checke the remaining "NA" values in the "Month" column
shark_clean.loc[shark_clean["Month"].isna(),["Date", "Year", "Month"]].head(60)

,Date,Year,Month
99,NaN,2025,<NA>
101,NaN,2025,<NA>
102,NaN,2025,<NA>
103,NaN,2025,<NA>
104,NaN,2025,<NA>
105,NaN,2025,<NA>
106,NaN,2025,<NA>
107,NaN,2025,<NA>
108,NaN,2025,<NA>
110,NaN,2025,<NA>


In [250]:
#Formating the remaining  NA values in the "Date" column as possible
shark_clean.loc[919, "Month"] = "June"
shark_clean.loc[2085, "Month"] = "January"  
shark_clean.loc[2761, "Month"] = "April"  
shark_clean.loc[3848, "Month"] = "December"  

In [251]:
shark_clean["Month"].value_counts(dropna=False)

Month
July         488
September    396
August       396
June         358
October      334
April        299
May          296
January      288
March        280
November     259
December     254
February     228
<NA>         122
Name: count, dtype: int64

In [252]:
shark_clean.loc[shark_clean["Month"].isna(),["Date", "Year", "Month"]]

,Date,Year,Month
99,NaN,2025,<NA>
101,NaN,2025,<NA>
102,NaN,2025,<NA>
103,NaN,2025,<NA>
104,NaN,2025,<NA>
...,...,...,...
3883,1980s,1980,<NA>
3908,1979,1979,<NA>
3933,1978,1978,<NA>
3934,1978,1978,<NA>


In [253]:
#See all existing Type categories
shark_clean["Type"].value_counts(dropna=False)

Type
Unprovoked             3162
Provoked                304
Invalid                 266
Watercraft              176
Sea Disaster             45
Questionable             27
NaN                      14
?                         1
Unconfirmed               1
Unverified                1
Under Investigation       1
Name: count, dtype: int64

In [254]:
#remove spaces around the values, standardize capitalization for type
shark_clean["Type"] = shark_clean["Type"].str.strip()
shark_clean["Type"] = shark_clean["Type"].str.title()

In [255]:
shark_clean["Type"].value_counts(dropna=False)

Type
Unprovoked             3162
Provoked                304
Invalid                 266
Watercraft              176
Sea Disaster             45
Questionable             27
NaN                      14
?                         1
Unconfirmed               1
Unverified                1
Under Investigation       1
Name: count, dtype: int64

In [256]:
shark_clean["Type"] = shark_clean["Type"].replace("?", pd.NA)

In [257]:
shark_clean["Type"].value_counts(dropna=False)

Type
Unprovoked             3162
Provoked                304
Invalid                 266
Watercraft              176
Sea Disaster             45
Questionable             27
NaN                      15
Unconfirmed               1
Unverified                1
Under Investigation       1
Name: count, dtype: int64

In [258]:
#Creating the final cleaned DataFrame with the selected columns "date", "year", "month", "type", 
cleaned_columns_2 = shark_clean[["Date", "Year", "Month", "Type"]].copy()

In [259]:
#Check
cleaned_columns_2.head()

,Date,Year,Month,Type
0,18 september,2026,September,Unprovoked
1,16 september,2026,September,Unprovoked
2,14 september,2026,September,Unprovoked
3,13 september,2026,September,Unprovoked
4,7 september,2026,September,Unprovoked


In [260]:
#check shape
cleaned_columns_2.shape

(3998, 4)

# Clean columns

In [261]:
shark_clean_tfs.head(10)

,Time,Species,Fatal Y/N
0,10:15,White Shark,Y
1,10:30,White Shark,N
2,17:30,Other/Unknown,N
3,09:45,Other/Unknown,N
4,16:40,Tiger Shark,N
5,11:00,Other/Unknown,N
6,16:35,Bull Shark,N
7,13:40,Other/Unknown,N
8,13:30,Tiger Shark,N
9,17:00,Other/Unknown,N


In [262]:
shark_clean_y.head(10)

,Sex,Age,Injury_Category
0,M,63.0,Other
1,M,NaN,Injury
2,F,37.0,Injury
3,M,NaN,Severe injury
4,M,24.0,Other
5,M,NaN,Injury
6,M,40.0,Injury
7,F,NaN,Injury
8,F,NaN,No injury
9,M,43.0,Injury


In [263]:
shark_clean.head(10)

,Country,State,Location,Activity,Date,Year,Type,Month
0,Australia,Western Australia,Sorrento Beach Perth,Swimming,18 september,2026,Unprovoked,September
1,Canada,Quebec,Off The Coast Of Perce Le Bilbo Dive Site,Diving,16 september,2026,Unprovoked,September
2,Bahamas,Bimini,Bimini Island,Swimming,14 september,2026,Unprovoked,September
3,Australia,Western Australia,Geraldton,Surfing,13 september,2026,Unprovoked,September
4,Usa,Hawaii,Honolulu,Surfing,7 september,2026,Unprovoked,September
5,Usa,Florida,New Smyrna Beach,Surfing,2 september,2026,Unprovoked,September
6,Usa,Florida,30 Miles Off Pensacola,Spearfishing,1 september,2026,Unprovoked,September
7,Usa,Massachusetts,Norton Point Edgartown Marthas Vinyard,Swimming,31 august,2026,Unprovoked,August
8,Usa,Hawaii,"Ala Moana Beach Park, Oahu",Surfing,26 august,2026,Unprovoked,August
9,Usa,Florida,Boca Grande (Island In The Keys),Wading on a sandbar,23 august,2026,Unprovoked,August


In [264]:
print(shark_clean_tfs.shape)
print(shark_clean_y.shape)
print(shark_clean.shape)
print(cleaned_columns_2.shape)

(3998, 3)
(3998, 3)
(3998, 8)
(3998, 4)


In [265]:
print(shark_clean_tfs.index.equals(shark_clean_y.index))
print(shark_clean_tfs.index.equals(shark_clean.index))
print(shark_clean_tfs.index.equals(cleaned_columns_2.index))

True
True
True


In [266]:
# Combine all cleaned columns into one final dataframe
shark_clean_final = pd.concat(
    [
        shark_clean,
        shark_clean_y,
        shark_clean_tfs
    ],
    axis=1
)

# Clean Final df

In [272]:
shark_clean_final.head(10)

,Country,State,Location,Activity,Date,Year,Type,Month,Sex,Age,Injury_Category,Time,Species,Fatal Y/N
0,Australia,Western Australia,Sorrento Beach Perth,Swimming,18 september,2026,Unprovoked,September,M,63.0,Other,10:15,White Shark,Y
1,Canada,Quebec,Off The Coast Of Perce Le Bilbo Dive Site,Diving,16 september,2026,Unprovoked,September,M,NaN,Injury,10:30,White Shark,N
2,Bahamas,Bimini,Bimini Island,Swimming,14 september,2026,Unprovoked,September,F,37.0,Injury,17:30,Other/Unknown,N
3,Australia,Western Australia,Geraldton,Surfing,13 september,2026,Unprovoked,September,M,NaN,Severe injury,09:45,Other/Unknown,N
4,Usa,Hawaii,Honolulu,Surfing,7 september,2026,Unprovoked,September,M,24.0,Other,16:40,Tiger Shark,N
5,Usa,Florida,New Smyrna Beach,Surfing,2 september,2026,Unprovoked,September,M,NaN,Injury,11:00,Other/Unknown,N
6,Usa,Florida,30 Miles Off Pensacola,Spearfishing,1 september,2026,Unprovoked,September,M,40.0,Injury,16:35,Bull Shark,N
7,Usa,Massachusetts,Norton Point Edgartown Marthas Vinyard,Swimming,31 august,2026,Unprovoked,August,F,NaN,Injury,13:40,Other/Unknown,N
8,Usa,Hawaii,"Ala Moana Beach Park, Oahu",Surfing,26 august,2026,Unprovoked,August,F,NaN,No injury,13:30,Tiger Shark,N
9,Usa,Florida,Boca Grande (Island In The Keys),Wading on a sandbar,23 august,2026,Unprovoked,August,M,43.0,Injury,17:00,Other/Unknown,N


In [273]:
shark_clean_final.to_csv("shark_clean_final.csv", index=False)

In [274]:
import os
os.getcwd()

'c:\\Users\\yurii\\Desktop'